<a href="https://colab.research.google.com/github/anshulk-cmu/llm-evaluation---housing/blob/main/Attribution_Graphs_Housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
Attribution Graphs - Housing
==============================
Qwen3-4B + mwhanna/qwen3-4b-transcoders
Tested on: RTX 6000 Pro (102GB VRAM)

WORKFLOW — run cells in order, one at a time:
  Cell 1  — Install circuit-tracer         (once per fresh session)
  Cell 2  — Fix numpy + RESTART RUNTIME    (once per fresh session)
  Cell 3  — Verify environment             (first cell after restart)
  Cell 4  — Load model + transcoders
  Cell 5  — Validate predictions
  Cell 6  — Run attribution, save graph, create JSON, free GPU
  Cell 7  — Load saved graph, inspect JSON structure
  Cell 8  — Analyze graph: bathroom vs bedroom contributions

CONFIRMED FACTS (validated through iterative debugging — do not change):
  - attribute() signature  : attribute(prompt, model)   <- prompt FIRST
  - create_graph_files sig : (graph, slug, output_path, node_threshold, edge_threshold)
                             NO model arg, NO prompt arg, NO pruning_threshold
  - Graph load             : torch.load(path, weights_only=False)
                             NOT Graph.from_pt() — crashes: 'Graph' has no attr 'get'
  - model_name             : "Qwen/Qwen3-4B"
  - transcoder_name        : "mwhanna/qwen3-4b-transcoders"
  - numpy must be >= 2.0 after install (circuit-tracer downgrades it; Cell 2 fixes it)
  - runtime restart required after numpy upgrade
  - Raw .pt graph is ~8GB — delete from memory after JSON is created
  - GPU: clear after attribution and before analysis with torch.cuda.empty_cache()

GRAPH OBJECT ATTRIBUTES (confirmed from inspection):
  activation_values, active_features, adjacency_matrix, cfg,
  input_string, input_tokens, logit_probabilities, logit_targets,
  logit_token_ids, logit_tokens, n_pos, scan, selected_features, vocab_size
"""


# ==============================================================================
# CELL 1 — Install circuit-tracer
# Run ONCE at the start of a fresh Colab session.
# Do NOT re-run after restarting the runtime — install persists.
# ==============================================================================

import subprocess, sys, torch

# Must clone and install from repo.
# plain `pip install circuit-tracer` installs a completely different package on PyPI.
!git clone https://github.com/safety-research/circuit-tracer.git
%cd circuit-tracer
!pip install -e . --quiet

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Enable GPU runtime before continuing.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {vram_gb:.1f} GB")
print("VRAM OK." if vram_gb >= 20 else "WARNING: < 20GB — add offload=True in Cell 4.")
print("\nCell 1 done. Proceed to Cell 2.")

Cloning into 'circuit-tracer'...
remote: Enumerating objects: 639, done.
remote: Counting objects: 100% (348/348), done.
remote: Compressing objects: 100% (219/219), done.
remote: Total 639 (delta 254), reused 129 (delta 129), pack-reused 291 (from 2)
Receiving objects: 100% (639/639), 2.62 MiB | 15.96 MiB/s, done.
Resolving deltas: 100% (370/370), done.
/content/circuit-tracer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.6 MB/s eta 0:00:00
   ━━━

In [2]:
# ==============================================================================
# CELL 2 — Fix numpy conflict, then RESTART RUNTIME
#
# circuit-tracer's deps downgrade numpy to 1.26.4.
# Colab's compiled C extensions need numpy >= 2.0.
# Mismatch error: ValueError: numpy.dtype size changed, binary incompatibility.
# Fix: upgrade numpy, restart so the correct binary loads.
# After restart: skip Cells 1 and 2, start directly from Cell 3.
# ==============================================================================

!pip install --upgrade "numpy>=2.0" --quiet

print("numpy upgraded.")
print("=" * 50)
print("ACTION REQUIRED: Runtime -> Restart session")
print("After restart: skip Cells 1 & 2, start from Cell 3.")
print("=" * 50)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 157.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformer-lens 2.17.0 requires numpy<2,>=1.26; python_version == "3.12", but you have numpy 2.4.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.
numpy upgraded.
ACTION REQUIRED: Runtime -> Restart session
After restart: skip Cells 1 & 2, start from Cell 3.


In [1]:
# ==============================================================================
# CELL 3 — Verify environment
# This is the FIRST cell to run after restarting the runtime.
# ==============================================================================

import sys, torch, numpy as np, urllib.request

sys.path.insert(0, '/content/circuit-tracer')

print(f"Python : {sys.version.split()[0]}")
print(f"numpy  : {np.__version__}  "
      f"{'OK' if int(np.__version__.split('.')[0]) >= 2 else 'FAIL — must be >= 2.0, re-run Cell 2'}")
print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

try:
    from circuit_tracer import ReplacementModel
    print("\ncircuit_tracer : OK")
except Exception as e:
    print(f"\ncircuit_tracer FAILED: {e}")
    print("Re-run Cell 1 (%cd circuit-tracer first), then restart again.")

try:
    urllib.request.urlopen("https://huggingface.co", timeout=5)
    print("HuggingFace    : reachable")
except Exception:
    print("HuggingFace    : NOT reachable — check Colab network settings")

print("\nCell 3 done. Proceed to Cell 4.")

Python : 3.12.12
numpy  : 2.4.3  OK
torch  : 2.10.0+cu128
CUDA   : True
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM   : 102.0 GB

circuit_tracer : OK
HuggingFace    : reachable

Cell 3 done. Proceed to Cell 4.


In [2]:
# ==============================================================================
# CELL 4 — Load Qwen3-4B + transcoders
#
# First run: downloads ~8-10 GB to ~/.cache/huggingface/  (~5-10 min)
# Subsequent runs: loads from cache                        (~30 sec)
# ==============================================================================

import sys, torch
sys.path.insert(0, '/content/circuit-tracer')

from circuit_tracer import ReplacementModel

torch.cuda.empty_cache()
print(f"VRAM before load : {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print("\nLoading Qwen3-4B + transcoders...")

model = ReplacementModel.from_pretrained(
    "Qwen/Qwen3-4B",                  # base model — completion-style prompts only
    "mwhanna/qwen3-4b-transcoders",   # PLT transcoders (mwhanna on HuggingFace)
    dtype=torch.bfloat16,
    # offload=True  <- uncomment only if VRAM < 20GB
)

print(f"\nModel loaded.")
print(f"  Backend  : {model.backend}")
print(f"  Device   : {model.cfg.device}")
print(f"  Layers   : {model.cfg.n_layers}")
print(f"  d_model  : {model.cfg.d_model}")
print(f"  VRAM now : {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
print(f"  VRAM free: "
      f"{(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB free")
print("\nCell 4 done. Proceed to Cell 5.")

VRAM before load : 0.00 GB allocated

Loading Qwen3-4B + transcoders...


config.yaml:   0%|          | 0.00/127 [00:00<?, ?B/s]

Fetching 36 files:   0%|          | 0/36 [00:00<?, ?it/s]

layer_13.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

layer_10.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen3-4B into HookedTransformer

Model loaded.
  Backend  : transformerlens
  Device   : cuda
  Layers   : 36
  d_model  : 2560
  VRAM now : 39.11 GB allocated
  VRAM free: 62.86 GB free

Cell 4 done. Proceed to Cell 5.


In [3]:
# ==============================================================================
# CELL 5 — Validate predictions on the conflict example
#
# Conflict case (zipcode 15220):
#   Property 1: 3 bathrooms, 3 bedrooms  -> ~$290K  <- correct
#   Property 2: 1 bathroom,  4 bedrooms  -> ~$156K
#
# Property 2 has more bedrooms but Property 1 has more bathrooms.
# Bathrooms are statistically more predictive of price.
# If the model picks Property 2, it is bedroom-brained.
# If it picks Property 1, the circuit should show bathrooms driving that choice.
#
# Prompt design:
#   - Base model -> completion-style only (no [INST] or system prompt)
#   - Trailing space is required: without it, next predicted token is a
#     whitespace token, not "1" or "2" directly
# ==============================================================================

import torch

CONFLICT_PROMPT = (
    "Property 1: 3 bathrooms, 3 bedrooms. "
    "Property 2: 1 bathroom, 4 bedrooms. "
    "The more expensive property is Property "
)

tokens = model.tokenizer.encode(CONFLICT_PROMPT)
print(f"Prompt      : {repr(CONFLICT_PROMPT)}")
print(f"Token count : {len(tokens)}  (no limit locally — Neuronpedia browser limit was 64)\n")

print("Tokenization (important for identifying positions in Cell 8 analysis):")
for i, tid in enumerate(tokens):
    s = model.tokenizer.decode([tid])
    print(f"  [{i:2d}]  id={tid:6d}  {repr(s)}")

with torch.no_grad():
    input_ids = model.tokenizer.encode(
        CONFLICT_PROMPT, return_tensors="pt"
    ).to(model.cfg.device)
    logits = model(input_ids)[0, -1, :]
    probs  = torch.softmax(logits, dim=-1)
    topk   = torch.topk(probs, 5)

del input_ids, logits
torch.cuda.empty_cache()

print("\nTop-5 next-token predictions:")
for prob, idx in zip(topk.values, topk.indices):
    tok = model.tokenizer.decode([idx.item()])
    print(f"  {repr(tok):12s}  p={prob.item():.4f}")

tok_1 = model.tokenizer.encode("1", add_special_tokens=False)[-1]
tok_2 = model.tokenizer.encode("2", add_special_tokens=False)[-1]
p1 = probs[tok_1].item()
p2 = probs[tok_2].item()

print(f"\nP('1') = {p1:.4f}  |  P('2') = {p2:.4f}")
if p1 > p2:
    print("Prediction: Property 1  CORRECT  -- bathrooms win")
elif p2 > p1:
    print("Prediction: Property 2  WRONG    -- bedroom-brained bias active")
else:
    print("Prediction: tie")

print(f"\nVRAM after Cell 5: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("\nCell 5 done. Proceed to Cell 6.")

Prompt      : 'Property 1: 3 bathrooms, 3 bedrooms. Property 2: 1 bathroom, 4 bedrooms. The more expensive property is Property '
Token count : 31  (no limit locally — Neuronpedia browser limit was 64)

Tokenization (important for identifying positions in Cell 8 analysis):
  [ 0]  id=  3052  'Property'
  [ 1]  id=   220  ' '
  [ 2]  id=    16  '1'
  [ 3]  id=    25  ':'
  [ 4]  id=   220  ' '
  [ 5]  id=    18  '3'
  [ 6]  id= 39883  ' bathrooms'
  [ 7]  id=    11  ','
  [ 8]  id=   220  ' '
  [ 9]  id=    18  '3'
  [10]  id= 27589  ' bedrooms'
  [11]  id=    13  '.'
  [12]  id=  8655  ' Property'
  [13]  id=   220  ' '
  [14]  id=    17  '2'
  [15]  id=    25  ':'
  [16]  id=   220  ' '
  [17]  id=    16  '1'
  [18]  id= 14852  ' bathroom'
  [19]  id=    11  ','
  [20]  id=   220  ' '
  [21]  id=    19  '4'
  [22]  id= 27589  ' bedrooms'
  [23]  id=    13  '.'
  [24]  id=   576  ' The'
  [25]  id=   803  ' more'
  [26]  id= 11392  ' expensive'
  [27]  id=  3343  ' property'
  [28]  id

In [4]:
import torch, os, gc
from circuit_tracer.attribution.attribute import attribute
from circuit_tracer.utils.create_graph_files import create_graph_files

GRAPH_DIR = "/content/graphs"
PT_PATH   = f"{GRAPH_DIR}/conflict_3bath3bed_vs_1bath4bed.pt"
JSON_DIR  = f"{GRAPH_DIR}/conflict_json"
os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)

CONFLICT_PROMPT = (
    "Property 1: 3 bathrooms, 3 bedrooms. "
    "Property 2: 1 bathroom, 4 bedrooms. "
    "The more expensive property is Property "
)

print(f"VRAM before attribution: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Running attribution...")
graph = attribute(CONFLICT_PROMPT, model)
print("Attribution complete.")
print(f"VRAM after attribution : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Save raw .pt
torch.save(graph, PT_PATH)
print(f"Raw graph saved: {PT_PATH}  ({os.path.getsize(PT_PATH) / 1e9:.2f} GB)")

# Free the 8 GB graph — JSON is all we need for Cell 8
del graph
gc.collect()
torch.cuda.empty_cache()
print(f"\nVRAM after freeing graph: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("\nCell 6 done. Proceed to Cell 7.")

VRAM before attribution: 39.12 GB
Running attribution...


sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


Attribution complete.
VRAM after attribution : 39.13 GB
Raw graph saved: /content/graphs/conflict_3bath3bed_vs_1bath4bed.pt  (8.44 GB)

VRAM after freeing graph: 39.13 GB

Cell 6 done. Proceed to Cell 7.


In [5]:
import torch, os, gc
from circuit_tracer.utils.create_graph_files import create_graph_files

GRAPH_DIR = "/content/graphs"
PT_PATH   = f"{GRAPH_DIR}/conflict_3bath3bed_vs_1bath4bed.pt"
JSON_DIR  = f"{GRAPH_DIR}/conflict_json"
os.makedirs(JSON_DIR, exist_ok=True)

# ── Patch find_threshold to sort on CPU (avoids the 15 GB GPU alloc) ──────────
import circuit_tracer.graph as _cg

def _find_threshold_cpu(scores: torch.Tensor, threshold: float):
    scores_cpu = scores.cpu().float()          # move off GPU before sort
    sorted_scores = torch.sort(scores_cpu, descending=True).values
    cumulative_score = torch.cumsum(sorted_scores, dim=0) / torch.sum(sorted_scores)
    threshold_index = int(torch.searchsorted(cumulative_score, threshold).item())
    return sorted_scores[threshold_index].item()

_cg.find_threshold = _find_threshold_cpu
print("find_threshold patched to use CPU sort.")

# ── Reload graph from disk (tensors load to CPU by default) ───────────────────
print(f"Reloading graph from {PT_PATH} ...")
graph = torch.load(PT_PATH, weights_only=False)
print(f"Graph reloaded. Type: {type(graph)}")
print(f"VRAM after reload: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# ── Create pruned JSON ────────────────────────────────────────────────────────
print("Creating pruned JSON...")
create_graph_files(
    graph,
    slug="conflict_3bath3bed_vs_1bath4bed",
    output_path=JSON_DIR,
    node_threshold=0.8,
    edge_threshold=0.98,
)
print(f"JSON files: {os.listdir(JSON_DIR)}")

del graph
gc.collect()
torch.cuda.empty_cache()
print(f"\nVRAM after freeing graph: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("\nCell 6b done. Proceed to Cell 7.")

find_threshold patched to use CPU sort.
Reloading graph from /content/graphs/conflict_3bath3bed_vs_1bath4bed.pt ...
Graph reloaded. Type: <class 'circuit_tracer.graph.Graph'>
VRAM after reload: 39.14 GB
Creating pruned JSON...
JSON files: ['graph-metadata.json', 'conflict_3bath3bed_vs_1bath4bed.json']

VRAM after freeing graph: 39.13 GB

Cell 6b done. Proceed to Cell 7.


In [6]:
# ==============================================================================
# CELL 7 — Load saved graph and inspect structure
#
# Reloads the .pt graph from disk so we can inspect field shapes before
# writing the analysis in Cell 8.
#
# CONFIRMED: use torch.load(path, weights_only=False)
#            NOT Graph.from_pt() -- that expects a dict, crashes with:
#            AttributeError: 'Graph' object has no attribute 'get'
# ==============================================================================

import torch, os, json, gc

GRAPH_DIR = "/content/graphs"
PT_PATH   = f"{GRAPH_DIR}/conflict_3bath3bed_vs_1bath4bed.pt"
JSON_DIR  = f"{GRAPH_DIR}/conflict_json"

torch.cuda.empty_cache()
print(f"VRAM before load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print(f"\nLoading graph from {PT_PATH}...")
graph = torch.load(PT_PATH, weights_only=False)
print(f"Graph loaded. Type: {type(graph)}")
print(f"Graph attrs: {[a for a in dir(graph) if not a.startswith('_')]}\n")

print(f"input_string    : {repr(graph.input_string)}")
print(f"input_tokens    : {graph.input_tokens}")
print(f"n_pos           : {graph.n_pos}")
print(f"vocab_size      : {graph.vocab_size}")
print(f"logit_tokens    : {graph.logit_tokens}")
print(f"logit_token_ids : {graph.logit_token_ids}")
print(f"logit_probs     : {graph.logit_probabilities}")
print(f"\nadjacency_matrix shape : {graph.adjacency_matrix.shape}")
print(f"active_features        : type={type(graph.active_features)}", end="")
if hasattr(graph.active_features, 'shape'):
    print(f"  shape={graph.active_features.shape}")
else:
    print()
print(f"selected_features      : type={type(graph.selected_features)}", end="")
if hasattr(graph.selected_features, 'shape'):
    print(f"  shape={graph.selected_features.shape}")
else:
    print(f"  value={graph.selected_features}")

print("\n--- JSON files ---")
for fname in sorted(os.listdir(JSON_DIR)):
    fpath = f"{JSON_DIR}/{fname}"
    size_kb = os.path.getsize(fpath) / 1e3
    with open(fpath) as fh:
        data = json.load(fh)
    print(f"\n{fname}  ({size_kb:.1f} KB)")
    if isinstance(data, dict):
        print(f"  Keys: {list(data.keys())}")
        for key in ["nodes", "edges", "qParams", "features", "metadata"]:
            if key in data:
                val = data[key]
                n = len(val) if isinstance(val, (list, dict)) else type(val).__name__
                print(f"  '{key}': {n} items")
                if isinstance(val, list) and len(val) > 0:
                    print(f"  First item keys: {list(val[0].keys()) if isinstance(val[0], dict) else val[0]}")
    elif isinstance(data, list):
        print(f"  List of {len(data)} items")
        if len(data) > 0:
            print(f"  First item: {data[0]}")

print("\nCell 7 done. Proceed to Cell 8.")

VRAM before load: 39.13 GB

Loading graph from /content/graphs/conflict_3bath3bed_vs_1bath4bed.pt...
Graph loaded. Type: <class 'circuit_tracer.graph.Graph'>
Graph attrs: ['activation_values', 'active_features', 'adjacency_matrix', 'cfg', 'from_pt', 'input_string', 'input_tokens', 'logit_probabilities', 'logit_targets', 'logit_token_ids', 'logit_tokens', 'n_pos', 'scan', 'selected_features', 'to', 'to_pt', 'vocab_size']

input_string    : '<|im_end|>Property 1: 3 bathrooms, 3 bedrooms. Property 2: 1 bathroom, 4 bedrooms. The more expensive property is Property '
input_tokens    : tensor([151645,   3052,    220,     16,     25,    220,     18,  39883,     11,
           220,     18,  27589,     13,   8655,    220,     17,     25,    220,
            16,  14852,     11,    220,     19,  27589,     13,    576,    803,
         11392,   3343,    374,   8655,    220], device='cuda:0')
n_pos           : 32
vocab_size      : 151643
logit_tokens    : tensor([16, 17], device='cuda:0')
logit_tok

/tmp/ipykernel_4356/4036828026.py:30: DeprecationWarning: logit_tokens property is deprecated. Use logit_token_ids property instead.
  print(f"logit_tokens    : {graph.logit_tokens}")



conflict_3bath3bed_vs_1bath4bed.json  (345598.8 KB)
  Keys: ['metadata', 'qParams', 'nodes', 'links']
  'nodes': 5271 items
  First item keys: ['node_id', 'feature', 'layer', 'ctx_idx', 'feature_type', 'token_prob', 'is_target_logit', 'run_idx', 'reverse_ctx_idx', 'jsNodeId', 'clerp', 'influence', 'activation']
  'qParams': 5 items
  'metadata': 7 items

graph-metadata.json  (0.9 KB)
  Keys: ['graphs']

Cell 7 done. Proceed to Cell 8.


In [8]:
# ==============================================================================
# CELL 8 — Analyze the attribution graph (JSON-based, accounts for prepended token)
# ==============================================================================

import json, os, gc
from collections import defaultdict
import torch

GRAPH_DIR = "/content/graphs"
PT_PATH   = f"{GRAPH_DIR}/conflict_3bath3bed_vs_1bath4bed.pt"
JSON_DIR  = f"{GRAPH_DIR}/conflict_json"
JSON_PATH = f"{JSON_DIR}/conflict_3bath3bed_vs_1bath4bed.json"

# Reload tokenizer
if 'model' in vars():
    tokenizer = model.tokenizer
else:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B", trust_remote_code=True)

# Load graph for the ground-truth token sequence (has the prepended token)
print("Reloading graph for token positions...")
graph = torch.load(PT_PATH, weights_only=False)
input_token_ids = graph.input_tokens.cpu().tolist()
n_pos           = graph.n_pos
logit_probs     = graph.logit_probabilities.cpu().float().tolist()
logit_tokens    = list(graph.logit_tokens)   # extract BEFORE del graph
del graph; gc.collect(); torch.cuda.empty_cache()
print(f"n_pos = {n_pos}  (includes prepended token)")

# Verify logit token ordering — don't assume index 0 = '1', index 1 = '2'
print(f"logit_tokens = {logit_tokens}")
try:
    idx_1 = logit_tokens.index('1')
    idx_2 = logit_tokens.index('2')
    logit_prob_1 = logit_probs[idx_1]
    logit_prob_2 = logit_probs[idx_2]
except ValueError:
    print("WARNING: '1' or '2' not in logit_tokens — model may not be predicting these tokens")
    logit_prob_1, logit_prob_2 = logit_probs[0], logit_probs[1]

# Load the pruned JSON
with open(JSON_PATH) as f:
    data = json.load(f)
nodes = data["nodes"]
links = data.get("links") or data.get("edges") or []
print(f"Pruned graph: {len(nodes)} nodes, {len(links)} links\n")


# ------------------------------------------------------------------
# PART 1: Decode token positions from the ACTUAL graph input_tokens
# ------------------------------------------------------------------
print("=" * 65)
print("PART 1: Token positions (from graph.input_tokens)")
print("=" * 65)

token_strings = [tokenizer.decode([tid]) for tid in input_token_ids]
for i, (tid, s) in enumerate(zip(input_token_ids, token_strings)):
    print(f"  [{i:2d}]  id={tid:7d}  {repr(s)}")

# Semantic positions — based on actual decoded strings
bath_ctx = [i for i, s in enumerate(token_strings) if 'bath' in s.lower()]
bed_ctx  = [i for i, s in enumerate(token_strings) if 'bed'  in s.lower()]
num_ctx  = [i for i, s in enumerate(token_strings) if s.strip() in ['1','2','3','4']]

print(f"\nBathroom ctx : {bath_ctx} -> {[repr(token_strings[i]) for i in bath_ctx]}")
print(f"Bedroom  ctx : {bed_ctx}  -> {[repr(token_strings[i]) for i in bed_ctx]}")
print(f"Number   ctx : {num_ctx}  -> {[repr(token_strings[i]) for i in num_ctx]}")


# ------------------------------------------------------------------
# PART 2: Node inventory
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("PART 2: Node inventory")
print("=" * 65)

type_counts = defaultdict(int)
for n in nodes:
    type_counts[n.get("feature_type", "unknown")] += 1
for ft, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ft:35s}: {count}")

logit_nodes = [n for n in nodes if n.get("is_target_logit")]
print(f"\nTarget logit nodes ({len(logit_nodes)}):")
for n in logit_nodes:
    feat    = n['feature']
    label   = tokenizer.decode([feat]) if feat < tokenizer.vocab_size else str(feat)
    inf_str = f"{n['influence']:.4f}" if n['influence'] is not None else "N/A (target node)"
    print(f"  node_id={n['node_id']}  feature={feat} ('{label}')  "
          f"influence={inf_str}  is_target={n['is_target_logit']}")

print(f"\nLogit probs from graph: P('1')={logit_prob_1:.4f}  P('2')={logit_prob_2:.4f}")


# ------------------------------------------------------------------
# PART 3: Influence by token position (ctx_idx)
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("PART 3: Total influence by token position (ctx_idx)")
print("=" * 65)
print("  influence > 0 = pushes toward '1' (Property 1 = CORRECT answer)")
print("  influence < 0 = pushes toward '2' (Property 2 = wrong)\n")

inf_by_ctx   = defaultdict(float)
count_by_ctx = defaultdict(int)
for n in nodes:
    if not n.get("is_target_logit"):
        ctx = n.get("ctx_idx")
        inf = n.get("influence")
        if ctx is not None and inf is not None:
            inf_by_ctx[ctx]   += float(inf)
            count_by_ctx[ctx] += 1

print(f"  {'ctx':>4}  {'token':>15}  {'total_influence':>16}  {'n':>5}  role")
print(f"  {'-'*4}  {'-'*15}  {'-'*16}  {'-'*5}  {'-'*25}")
for ctx in sorted(inf_by_ctx.keys()):
    tok  = repr(token_strings[ctx]) if ctx < len(token_strings) else "?"
    inf  = inf_by_ctx[ctx]
    n    = count_by_ctx[ctx]
    role = ""
    if ctx in bath_ctx:  role = "<-- BATHROOM"
    elif ctx in bed_ctx: role = "<-- BEDROOM"
    elif ctx in num_ctx: role = "<-- NUMBER"
    direction = "→ Prop1 CORRECT" if inf > 0 else "→ Prop2 WRONG"
    print(f"  {ctx:>4}  {tok:>15}  {inf:>+16.4f}  {n:>5}  {direction}  {role}")


# ------------------------------------------------------------------
# PART 4: Aggregated by semantic group
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("PART 4: Aggregated influence by semantic group")
print("=" * 65)

# Find where "Property 2" starts — split bath/bed tokens before/after it
p2_start = next((i for i, s in enumerate(token_strings) if 'Property' in s
                 and i > 5), 14)   # fallback to 14 if not found
p1_bath = [c for c in bath_ctx if c < p2_start]
p2_bath = [c for c in bath_ctx if c >= p2_start]
p1_bed  = [c for c in bed_ctx  if c < p2_start]
p2_bed  = [c for c in bed_ctx  if c >= p2_start]
print(f"P1/P2 split at ctx {p2_start}  (token: {repr(token_strings[p2_start])})")

groups = {
    "P1 bathroom token(s)"   : p1_bath,
    "P1 bedroom  token(s)"   : p1_bed,
    "P2 bathroom token(s)"   : p2_bath,
    "P2 bedroom  token(s)"   : p2_bed,
    "Number tokens (1/2/3/4)": num_ctx,
    "All other tokens"        : [c for c in inf_by_ctx
                                  if c not in bath_ctx + bed_ctx + num_ctx],
}
for label, ctxs in groups.items():
    total = sum(inf_by_ctx[c] for c in ctxs if c in inf_by_ctx)
    toks  = [repr(token_strings[c]) for c in ctxs if c < len(token_strings)]
    direction = "→ Property 1 CORRECT" if total > 0 else "→ Property 2 WRONG"
    print(f"  {label:28s}: {total:>+8.4f}  {direction}  {toks}")


# ------------------------------------------------------------------
# PART 5: Top 25 nodes by |influence|
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("PART 5: Top 25 nodes by |influence|")
print("=" * 65)

non_logit = [n for n in nodes if not n.get("is_target_logit") and n.get("influence") is not None]
top_nodes = sorted(non_logit, key=lambda n: abs(float(n["influence"])), reverse=True)[:25]

print(f"  {'node_id':>22}  {'layer':>5}  {'ctx':>3}  {'token':>12}  "
      f"{'influence':>10}  {'activation':>10}  clerp")
print(f"  {'-'*22}  {'-'*5}  {'-'*3}  {'-'*12}  {'-'*10}  {'-'*10}  {'-'*35}")
for n in top_nodes:
    ctx   = n.get("ctx_idx", -1)
    tok   = repr(token_strings[ctx]) if 0 <= ctx < len(token_strings) else "?"
    clerp = n.get("clerp") or ""
    role  = ""
    if ctx in bath_ctx: role = "[BATH]"
    elif ctx in bed_ctx: role = "[BED]"
    print(f"  {n['node_id']:>22}  {str(n['layer']):>5}  {ctx:>3}  {tok:>12}  "
          f"  {float(n['influence']):>+9.4f}  {(float(n['activation']) if n['activation'] is not None else float('nan')):>10.4f}"
          f"  {role} {clerp[:50]}")


# ------------------------------------------------------------------
# PART 6: Influence by layer
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("PART 6: Influence by layer (where does computation happen?)")
print("=" * 65)

inf_abs_layer = defaultdict(float)
inf_pos_layer = defaultdict(float)
inf_neg_layer = defaultdict(float)
for n in non_logit:
    layer = n.get("layer", "?")
    inf   = float(n["influence"])
    inf_abs_layer[layer] += abs(inf)
    if inf > 0: inf_pos_layer[layer] += inf
    else:       inf_neg_layer[layer] += inf

def layer_key(k):
    try: return int(k)
    except: return 999

print(f"  {'layer':>6}  {'|influence|':>12}  {'positive':>10}  {'negative':>10}")
print(f"  {'-'*6}  {'-'*12}  {'-'*10}  {'-'*10}")
for layer in sorted(inf_abs_layer.keys(), key=layer_key):
    print(f"  {str(layer):>6}  {inf_abs_layer[layer]:>12.4f}  "
          f"{inf_pos_layer[layer]:>+10.4f}  {inf_neg_layer[layer]:>+10.4f}")


# ------------------------------------------------------------------
# PART 7: Neuronpedia lookup list
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("PART 7: Top features for Neuronpedia lookup")
print("=" * 65)
print("URL: https://www.neuronpedia.org/qwen3-4b")
print()

seen = set()
for n in top_nodes:
    if n.get("feature_type") in ("embedding", "logit"):
        continue
    key = (n.get("layer"), n.get("feature"))
    if key in seen:
        continue
    seen.add(key)
    ctx   = n.get("ctx_idx", -1)
    tok   = repr(token_strings[ctx]) if 0 <= ctx < len(token_strings) else "?"
    role  = "[BATH]" if ctx in bath_ctx else "[BED]" if ctx in bed_ctx else ""
    clerp = n.get("clerp") or "(no label cached)"
    print(f"  Layer {str(n['layer']):>3}  Feature {n['feature']:>8}  "
          f"inf={float(n['influence']):>+7.4f}  tok={tok:>12}  {role}  {clerp[:55]}")


# ------------------------------------------------------------------
# PART 8: Core research summary
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("PART 8: Research summary")
print("=" * 65)

bath_total = sum(inf_by_ctx.get(c, 0) for c in bath_ctx)
bed_total  = sum(inf_by_ctx.get(c, 0) for c in bed_ctx)
num_total  = sum(inf_by_ctx.get(c, 0) for c in num_ctx)

print(f"  Model prediction  : P('1')={logit_prob_1:.4f}  P('2')={logit_prob_2:.4f}")
print(f"  Correct answer    : Property 1 (3 bath / 3 bed, ~$290K)")
print(f"  Prediction        : {'CORRECT' if logit_prob_1 > logit_prob_2 else 'WRONG'}")
print()
print(f"  Bathroom influence (all): {bath_total:>+8.4f}  "
      f"{'CORRECT direction →P1' if bath_total > 0 else 'WRONG direction →P2'}")
print(f"  Bedroom  influence (all): {bed_total:>+8.4f}  "
      f"{'pushes →P1' if bed_total > 0 else 'pushes →P2'}")
print(f"  Number   influence (all): {num_total:>+8.4f}")
print()
print("  Q: Do bathroom tokens drive the correct answer?")
print(f"     -> {'YES' if bath_total > 0 else 'NO'} (influence = {bath_total:+.4f})")
print()
print("  Q: Do bedroom tokens fight against the correct answer?")
print(f"     -> {'YES — bedroom bias confirmed' if bed_total < 0 else 'NO — bedroom does not oppose bathrooms here'}")
print(f"        (influence = {bed_total:+.4f})")
print()
print("  Q: Are decisive features real-estate concepts or generic text patterns?")
print("     -> Check clerp labels in Parts 5 & 7.")
print("        If all empty/generic: same finding as Gemma-2-2B.")
print("        If real-estate labels appear: Qwen3-4B differs from Gemma-2-2B.")

Reloading graph for token positions...


/tmp/ipykernel_4356/3439791304.py:27: DeprecationWarning: logit_tokens property is deprecated. Use logit_token_ids property instead.
  logit_tokens    = list(graph.logit_tokens)   # extract BEFORE del graph


n_pos = 32  (includes prepended token)
logit_tokens = [tensor(16, device='cuda:0'), tensor(17, device='cuda:0')]
Pruned graph: 5271 nodes, 3265350 links

PART 1: Token positions (from graph.input_tokens)
  [ 0]  id= 151645  '<|im_end|>'
  [ 1]  id=   3052  'Property'
  [ 2]  id=    220  ' '
  [ 3]  id=     16  '1'
  [ 4]  id=     25  ':'
  [ 5]  id=    220  ' '
  [ 6]  id=     18  '3'
  [ 7]  id=  39883  ' bathrooms'
  [ 8]  id=     11  ','
  [ 9]  id=    220  ' '
  [10]  id=     18  '3'
  [11]  id=  27589  ' bedrooms'
  [12]  id=     13  '.'
  [13]  id=   8655  ' Property'
  [14]  id=    220  ' '
  [15]  id=     17  '2'
  [16]  id=     25  ':'
  [17]  id=    220  ' '
  [18]  id=     16  '1'
  [19]  id=  14852  ' bathroom'
  [20]  id=     11  ','
  [21]  id=    220  ' '
  [22]  id=     19  '4'
  [23]  id=  27589  ' bedrooms'
  [24]  id=     13  '.'
  [25]  id=    576  ' The'
  [26]  id=    803  ' more'
  [27]  id=  11392  ' expensive'
  [28]  id=   3343  ' property'
  [29]  id=    374 

In [9]:
# PART 9 (corrected): adj is [target, source] — logits are ROWS not columns
import torch, gc
from collections import defaultdict

# Ensure token_strings and semantic ctx sets are defined (may come from Cell 8)
if 'token_strings' not in vars():
    print("Recomputing token positions from graph (Cell 8 vars not in memory)...")
    _g_tmp = torch.load(PT_PATH, weights_only=False)
    _input_token_ids = _g_tmp.input_tokens.cpu().tolist()
    del _g_tmp; gc.collect()
    from transformers import AutoTokenizer
    _tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B", trust_remote_code=True)
    token_strings = [_tok.decode([tid]) for tid in _input_token_ids]
    bath_ctx = [i for i, s in enumerate(token_strings) if 'bath' in s.lower()]
    bed_ctx  = [i for i, s in enumerate(token_strings) if 'bed'  in s.lower()]
    num_ctx  = [i for i, s in enumerate(token_strings) if s.strip() in ['1','2','3','4']]
    print(f"  bath_ctx={bath_ctx}, bed_ctx={bed_ctx}, num_ctx={num_ctx}")

graph     = torch.load(PT_PATH, weights_only=False)
adj       = graph.adjacency_matrix.cpu().float()
n_pos     = graph.n_pos
af        = graph.active_features.cpu()  # [N_features, 3]: (layer, feat_idx, ctx_idx)

# Logit nodes are the last two ROWS (confirmed: rows 45917/45918 each have 45516 non-zeros)
logit_row_1 = adj.shape[0] - 2   # row for token '1'
logit_row_2 = adj.shape[0] - 1   # row for token '2'
print(f"adj shape     : {adj.shape}")
print(f"Logit row '1' : {logit_row_1}  (non-zeros: {int((adj[logit_row_1] != 0).sum())})")
print(f"Logit row '2' : {logit_row_2}  (non-zeros: {int((adj[logit_row_2] != 0).sum())})")

# differential[col] = how much source node 'col' favors '1' over '2'
diff_vec = adj[logit_row_1] - adj[logit_row_2]   # shape [45919]
print(f"\ndiff_vec non-zero entries : {int((diff_vec != 0).sum())}")
print(f"diff_vec range: min={float(diff_vec.min()):.6f}  max={float(diff_vec.max()):.6f}")

# ── Input token positions (cols 0..n_pos-1) ────────────────────────────────
print(f"\n--- Direct input token → logit differential ---")
print(f"  {'ctx':>4}  {'token':>14}  {'->logit1':>10}  {'->logit2':>10}  {'diff':>10}  role")
print(f"  {'-'*4}  {'-'*14}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*15}")
for pos in range(n_pos):
    e1   = float(adj[logit_row_1, pos])
    e2   = float(adj[logit_row_2, pos])
    diff = e1 - e2
    tok  = repr(token_strings[pos]) if pos < len(token_strings) else "?"
    role = "<-- BATH" if pos in bath_ctx else ("<-- BED" if pos in bed_ctx else
           ("<-- NUM" if pos in num_ctx else ""))
    if abs(diff) > 1e-7 or abs(e1) > 1e-7:
        print(f"  {pos:>4}  {tok:>14}  {e1:>10.6f}  {e2:>10.6f}  {diff:>+10.6f}  {role}")

# ── Feature nodes (cols n_pos .. 45916) ────────────────────────────────────
print(f"\n--- Feature node → logit differential ---")
print(f"  Positive = pushes toward '1' (CORRECT), Negative = toward '2' (WRONG)\n")

diff_by_ctx  = defaultdict(float)
rows_by_ctx  = defaultdict(list)
feature_diffs = []

for col in range(n_pos, adj.shape[1] - 2):   # exclude the two logit rows themselves
    e1   = float(adj[logit_row_1, col])
    e2   = float(adj[logit_row_2, col])
    diff = e1 - e2
    if abs(diff) < 1e-7: continue            # skip zero entries

    fi  = col - n_pos
    if fi >= len(af): continue
    layer = int(af[fi, 0])
    ctx   = int(af[fi, 2])
    tok   = repr(token_strings[ctx]) if ctx < len(token_strings) else "?"
    role  = "<-- BATH" if ctx in bath_ctx else ("<-- BED" if ctx in bed_ctx else
            ("<-- NUM" if ctx in num_ctx else ""))
    diff_by_ctx[ctx] += diff
    rows_by_ctx[ctx].append(col)
    feature_diffs.append((col, layer, ctx, tok, e1, e2, diff, role))

print(f"Features with non-zero differential: {len(feature_diffs)}")

# ── Aggregate by ctx_idx ───────────────────────────────────────────────────
print(f"\nNet differential by token position (aggregated over all features at that ctx):")
print(f"  {'ctx':>4}  {'token':>14}  {'net_diff':>12}  {'n_feats':>7}  role  verdict")
print(f"  {'-'*4}  {'-'*14}  {'-'*12}  {'-'*7}  {'-'*8}  {'-'*22}")
for ctx in sorted(diff_by_ctx.keys()):
    tok  = repr(token_strings[ctx]) if ctx < len(token_strings) else "?"
    d    = diff_by_ctx[ctx]
    n    = len(rows_by_ctx[ctx])
    role = "BATH" if ctx in bath_ctx else ("BED" if ctx in bed_ctx else
           ("NUM"  if ctx in num_ctx  else ""))
    verdict = "→ P1 CORRECT ✓" if d > 1e-6 else ("→ P2 WRONG ✗" if d < -1e-6 else "~neutral")
    print(f"  {ctx:>4}  {tok:>14}  {d:>+12.6f}  {n:>7}  {role:>6}  {verdict}")

# ── Semantic group aggregation ─────────────────────────────────────────────
print(f"\nSemantic group aggregation:")
print(f"  {'group':38s}  {'net_diff':>12}  verdict")
print(f"  {'-'*38}  {'-'*12}  {'-'*25}")
groups_d = {
    "P1 ' bathrooms' (ctx  7)"  : [7],
    "P1 ' bedrooms'  (ctx 11)"  : [11],
    "P2 ' bathroom'  (ctx 19)"  : [19],
    "P2 ' bedrooms'  (ctx 23)"  : [23],
    "All bathroom tokens"       : bath_ctx,
    "All bedroom  tokens"       : bed_ctx,
    "Number tokens"             : num_ctx,
    "Structural / other"        : [c for c in diff_by_ctx
                                    if c not in bath_ctx+bed_ctx+num_ctx],
}
for label, ctxs in groups_d.items():
    total = sum(diff_by_ctx.get(c, 0.0) for c in ctxs)
    verdict = "→ P1 CORRECT ✓" if total > 1e-6 else \
              ("→ P2 WRONG ✗"  if total < -1e-6 else "~neutral")
    print(f"  {label:38s}  {total:>+12.6f}  {verdict}")

# ── Top 20 individual features by |differential| ──────────────────────────
print(f"\nTop 20 individual features by |differential|:")
print(f"  {'col':>6}  {'layer':>5}  {'ctx':>4}  {'token':>14}  {'diff':>10}  role")
print(f"  {'-'*6}  {'-'*5}  {'-'*4}  {'-'*14}  {'-'*10}  {'-'*8}")
for col, layer, ctx, tok, e1, e2, diff, role in sorted(
        feature_diffs, key=lambda x: abs(x[6]), reverse=True)[:20]:
    print(f"  {col:>6}  {layer:>5}  {ctx:>4}  {tok:>14}  {diff:>+10.6f}  {role}")

del graph; gc.collect(); torch.cuda.empty_cache()
print(f"\nVRAM after Part 9: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Streaming output truncated to the last 5000 lines.
  135062               ?     +0.001093        2          → P1 CORRECT ✓
  135067               ?     -0.000001        1          ~neutral
  135068               ?     +0.000255        2          → P1 CORRECT ✓
  135072               ?     -0.000005        1          → P2 WRONG ✗
  135087               ?     +0.000027        2          → P1 CORRECT ✓
  135098               ?     -0.000003        1          → P2 WRONG ✗
  135103               ?     +0.000618        1          → P1 CORRECT ✓
  135111               ?     -0.000301        1          → P2 WRONG ✗
  135136               ?     -0.000011        1          → P2 WRONG ✗
  135137               ?     +0.000162        1          → P1 CORRECT ✓
  135139               ?     -0.000062        1          → P2 WRONG ✗
  135144               ?     -0.000034        1          → P2 WRONG ✗
  135148               ?     -0.000256        1          → P2 WRONG ✗
  135152               ?     +0.0

In [16]:
# ============================================================
# CELL 10a — DIAGNOSTIC: Inspect Graph Files
# ============================================================

import json, torch, os
from pathlib import Path

JSON_PATH  = "/content/graphs/conflict_json/conflict_3bath3bed_vs_1bath4bed.json"
META_PATH  = "/content/graphs/conflict_json/graph-metadata.json"
PT_PATH    = "/content/graphs/conflict_3bath3bed_vs_1bath4bed.pt"

# ──────────────────────────────────────────────────────────────
# FILE SIZES
# ──────────────────────────────────────────────────────────────
for p in [JSON_PATH, META_PATH, PT_PATH]:
    size = Path(p).stat().st_size / 1e6
    print(f"📁 {Path(p).name:55s} {size:.1f} MB")

print("\n" + "="*70)

# ──────────────────────────────────────────────────────────────
# METADATA JSON
# ──────────────────────────────────────────────────────────────
print("\n── graph-metadata.json (full contents) ──")
with open(META_PATH) as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

# ──────────────────────────────────────────────────────────────
# MAIN JSON — top-level keys + shape
# ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("\n── conflict JSON: top-level keys ──")
with open(JSON_PATH) as f:
    graph = json.load(f)

print("Top-level keys:", list(graph.keys()))
for k, v in graph.items():
    if isinstance(v, list):
        print(f"  '{k}': list of {len(v):,} items")
    elif isinstance(v, dict):
        print(f"  '{k}': dict with keys {list(v.keys())[:8]}")
    else:
        print(f"  '{k}': {type(v).__name__} = {v}")

# ──────────────────────────────────────────────────────────────
# FIRST 3 NODES — every key/value
# ──────────────────────────────────────────────────────────────
node_key = next((k for k in graph if isinstance(graph[k], list) and "node" in k.lower()), None)
node_key = node_key or next((k for k in graph if isinstance(graph[k], list)), None)

print(f"\n── First 3 items in '{node_key}' ──")
for i, item in enumerate(graph[node_key][:3]):
    print(f"\n  Node {i}:")
    if isinstance(item, dict):
        for kk, vv in item.items():
            print(f"    {kk!r:25s}: {type(vv).__name__:10s} = {str(vv)[:80]}")
    else:
        print(f"    raw value: {item}")

# ──────────────────────────────────────────────────────────────
# FIRST 3 EDGES — every key/value
# ──────────────────────────────────────────────────────────────
edge_key = next((k for k in graph if isinstance(graph[k], list) and k != node_key), None)
if edge_key:
    print(f"\n── First 3 items in '{edge_key}' ──")
    for i, item in enumerate(graph[edge_key][:3]):
        print(f"\n  Edge {i}:")
        if isinstance(item, dict):
            for kk, vv in item.items():
                print(f"    {kk!r:25s}: {type(vv).__name__:10s} = {str(vv)[:80]}")
        else:
            print(f"    raw value: {item}")

# ──────────────────────────────────────────────────────────────
# UNIQUE KEYS across all nodes (sample 500)
# ──────────────────────────────────────────────────────────────
all_node_keys = set()
for item in graph[node_key][:500]:
    if isinstance(item, dict):
        all_node_keys.update(item.keys())
print(f"\n── All unique keys found across first 500 nodes ──")
print(sorted(all_node_keys))

# ──────────────────────────────────────────────────────────────
# NULL/NONE audit on first 500 nodes
# ──────────────────────────────────────────────────────────────
print(f"\n── None/null audit (first 500 nodes) ──")
null_counts = {}
for item in graph[node_key][:500]:
    if isinstance(item, dict):
        for kk, vv in item.items():
            if vv is None:
                null_counts[kk] = null_counts.get(kk, 0) + 1
if null_counts:
    for kk, cnt in sorted(null_counts.items(), key=lambda x: -x[1]):
        print(f"  '{kk}': {cnt} nulls out of 500")
else:
    print("  No None values found in first 500 nodes ✅")

# ──────────────────────────────────────────────────────────────
# PT FILE — structure
# ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("\n── .pt file structure ──")
pt = torch.load(PT_PATH, map_location="cpu", weights_only=False)

def describe(obj, depth=0, max_depth=3, prefix=""):
    indent = "  " * depth
    if depth > max_depth:
        print(f"{indent}{prefix}... (truncated)")
        return
    if isinstance(obj, dict):
        print(f"{indent}{prefix}dict({len(obj)} keys): {list(obj.keys())[:8]}")
        for k, v in list(obj.items())[:5]:
            describe(v, depth+1, max_depth, prefix=f"['{k}']: ")
    elif isinstance(obj, (list, tuple)):
        print(f"{indent}{prefix}{type(obj).__name__}({len(obj)} items)")
        if len(obj) > 0:
            describe(obj[0], depth+1, max_depth, prefix="[0]: ")
    elif isinstance(obj, torch.Tensor):
        print(f"{indent}{prefix}Tensor shape={list(obj.shape)} dtype={obj.dtype} "
              f"min={obj.min():.4f} max={obj.max():.4f} mean={obj.mean():.4f}")
    else:
        print(f"{indent}{prefix}{type(obj).__name__} = {str(obj)[:80]}")

describe(pt)

print("\n✅ Diagnostic complete.")

📁 conflict_3bath3bed_vs_1bath4bed.json                    345.6 MB
📁 graph-metadata.json                                     0.0 MB
📁 conflict_3bath3bed_vs_1bath4bed.pt                      8435.7 MB


── graph-metadata.json (full contents) ──
{
  "graphs": [
    {
      "slug": "conflict_3bath3bed_vs_1bath4bed",
      "scan": "mwhanna/qwen3-4b-transcoders",
      "transcoder_list": [],
      "prompt_tokens": [
        "<|im_end|>",
        "Property",
        " ",
        "1",
        ":",
        " ",
        "3",
        " bathrooms",
        ",",
        " ",
        "3",
        " bedrooms",
        ".",
        " Property",
        " ",
        "2",
        ":",
        " ",
        "1",
        " bathroom",
        ",",
        " ",
        "4",
        " bedrooms",
        ".",
        " The",
        " more",
        " expensive",
        " property",
        " is",
        " Property",
        " "
      ],
      "prompt": "<|im_end|>Property 1: 3 bathrooms, 3 bedrooms. Proper

In [35]:
# ============================================================
# CELL 10 — Attribution Graph Visualizer
# Qwen3-4B | Property Comparison | Conflict Case
# ============================================================

import json, re, collections
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# ── 0. CONFIG ────────────────────────────────────────────────
JSON_PATH = "/content/graphs/conflict_json/conflict_3bath3bed_vs_1bath4bed.json"
META_PATH = "/content/graphs/conflict_json/graph-metadata.json"

# Token map (ctx_idx → display label + semantic group)
TOKEN_MAP = {
    0:  ("<|im_end|>",  "structural"),
    1:  ("Property",    "structural"),
    2:  (" ",           "structural"),
    3:  ("1",           "p1_id"),
    4:  (":",           "structural"),
    5:  (" ",           "structural"),
    6:  ("3",           "p1_number"),
    7:  (" bathrooms",  "p1_bath"),
    8:  (",",           "structural"),
    9:  (" ",           "structural"),
    10: ("3",           "p1_number"),
    11: (" bedrooms",   "p1_bed"),
    12: (".",           "structural"),
    13: (" Property",   "structural"),
    14: (" ",           "structural"),
    15: ("2",           "p2_id"),
    16: (":",           "structural"),
    17: (" ",           "structural"),
    18: ("1",           "p2_number"),
    19: (" bathroom",   "p2_bath"),
    20: (",",           "structural"),
    21: (" ",           "structural"),
    22: ("4",           "p2_number"),
    23: (" bedrooms",   "p2_bed"),
    24: (".",           "structural"),
    25: (" The",        "structural"),
    26: (" more",       "structural"),
    27: (" expensive",  "structural"),
    28: (" property",   "structural"),
    29: (" is",         "structural"),
    30: (" Property",   "structural"),
    31: (" ",           "output"),
}

GROUP_COLORS = {
    "p1_bath":    "#00d4ff",
    "p1_bed":     "#0066ff",
    "p1_number":  "#4499ff",
    "p1_id":      "#88bbff",
    "p2_bath":    "#ff6600",
    "p2_bed":     "#ff0044",
    "p2_number":  "#ff8833",
    "p2_id":      "#ffaa66",
    "structural": "#888899",
    "output":     "#44ff88",
    "logit":      "#ffff00",
    "embedding":  "#cc88ff",
}

GROUP_LABELS = {
    "p1_bath":    "P1 Bathrooms (3)",
    "p1_bed":     "P1 Bedrooms (3)",
    "p1_number":  "P1 Numbers",
    "p1_id":      "P1 Identity",
    "p2_bath":    "P2 Bathroom (1)",
    "p2_bed":     "P2 Bedrooms (4)",
    "p2_number":  "P2 Numbers",
    "p2_id":      "P2 Identity",
    "structural": "Structural Tokens",
    "output":     "Output Position",
    "logit":      "Logit Node",
    "embedding":  "Embedding",
}

print("⏳ Loading JSON graph (~345MB)...")
with open(JSON_PATH) as f:
    data = json.load(f)
with open(META_PATH) as f:
    meta = json.load(f)

nodes_raw = data["nodes"]
links_raw = data["links"]
prompt_tokens = meta["graphs"][0]["prompt_tokens"]

print(f"✅ Loaded: {len(nodes_raw):,} nodes | {len(links_raw):,} links")

# ── 1. BUILD NODE DATAFRAME ──────────────────────────────────
def parse_node(n):
    ft = n["feature_type"]
    layer = int(n["layer"]) if str(n["layer"]).isdigit() else -1
    ctx   = int(n["ctx_idx"])
    tok_label, group = TOKEN_MAP.get(ctx, ("?", "structural"))
    if n.get("is_target_logit"):
        group = "logit"
    if ft == "embedding":
        group = "embedding"
    return {
        "node_id":      n["node_id"],
        "layer":        layer,
        "ctx_idx":      ctx,
        "feature":      n["feature"],
        "feature_type": ft,
        "influence":    float(n["influence"]) if n["influence"] is not None else 0.0,
        "activation":   float(n["activation"]) if n["activation"] is not None else 0.0,
        "clerp":        n.get("clerp", ""),
        "token":        tok_label,
        "group":        group,
        "color":        GROUP_COLORS.get(group, "#aaaaaa"),
        "is_logit":     bool(n.get("is_target_logit", False)),
    }

df_nodes = pd.DataFrame([parse_node(n) for n in nodes_raw])
df_links = pd.DataFrame(links_raw)

print(f"\n📊 Node type breakdown:")
print(df_nodes.groupby("feature_type")["node_id"].count().to_string())
print(f"\n📊 Semantic group breakdown:")
grp = df_nodes.groupby("group")["influence"].agg(["count","sum","mean"]).round(3)
grp.columns = ["count","total_influence","mean_influence"]
grp = grp.sort_values("total_influence", ascending=False)
print(grp.to_string())

# ── 2. HELPERS ───────────────────────────────────────────────
def top_nodes(n=200, by="influence"):
    return df_nodes.nlargest(n, by)

def build_nx_subgraph(node_ids_set, max_edges=5000):
    G = nx.DiGraph()
    id_df = df_nodes[df_nodes.node_id.isin(node_ids_set)]
    for _, row in id_df.iterrows():
        G.add_node(row.node_id, **row.to_dict())
    edge_df = df_links[
        df_links.source.isin(node_ids_set) &
        df_links.target.isin(node_ids_set)
    ].nlargest(max_edges, "weight")
    for _, e in edge_df.iterrows():
        G.add_edge(e.source, e.target, weight=float(e.weight))
    return G

def plotly_network(G, title, node_size_col="influence", layout="spring", height=750):
    if layout == "spring":
        pos = nx.spring_layout(G, seed=42, k=1.2)
    elif layout == "shell":
        pos = nx.shell_layout(G)
    else:
        pos = nx.kamada_kawai_layout(G)

    edge_x, edge_y, edge_colors = [], [], []
    for u, v, d in G.edges(data=True):
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
        edge_colors.append("#00ff88" if d["weight"] > 0 else "#ff4444")

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(width=0.6, color="#334455"),
        hoverinfo="none", opacity=0.5
    )

    node_x, node_y, node_text, node_colors, node_sizes = [], [], [], [], []
    for nid in G.nodes():
        x, y = pos[nid]
        nd = G.nodes[nid]
        node_x.append(x); node_y.append(y)
        infl = float(nd.get("influence", 0))
        node_sizes.append(max(6, min(30, infl * 15)))
        node_colors.append(nd.get("color", "#aaaaaa"))
        tok = nd.get("token", "?")
        lyr = nd.get("layer", "?")
        clp = nd.get("clerp", "") or "—"
        node_text.append(
            f"<b>{tok}</b><br>Layer {lyr} | ctx {nd.get('ctx_idx','?')}<br>"
            f"Influence: {infl:.4f}<br>Activation: {float(nd.get('activation',0)):.4f}<br>"
            f"Clerp: {clp}<br>Feature: {nd.get('feature','?')}"
        )

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers",
        marker=dict(color=node_colors, size=node_sizes,
                    line=dict(width=1, color="#ffffff")),
        text=node_text, hoverinfo="text"
    )

    fig = go.Figure([edge_trace, node_trace],
        layout=go.Layout(
            title=dict(text=title, font=dict(size=16, color="#e0e8ff")),
            paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
            height=height, showlegend=False,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            margin=dict(l=10, r=10, t=50, b=10),
            font=dict(color="#c0ccdd"),
        )
    )
    return fig

# ════════════════════════════════════════════════════════════
# VIZ 1 — SEMANTIC GROUP SUMMARY BAR
# ════════════════════════════════════════════════════════════
print("\n🎨 Rendering Viz 1: Semantic Group Influence...")

grp_data = df_nodes.groupby("group")["influence"].sum().reset_index()
grp_data["label"] = grp_data["group"].map(GROUP_LABELS)
grp_data["color"] = grp_data["group"].map(GROUP_COLORS)
grp_data = grp_data.sort_values("influence", ascending=True)

fig1 = go.Figure(go.Bar(
    x=grp_data["influence"], y=grp_data["label"],
    orientation="h",
    marker=dict(color=grp_data["color"], line=dict(width=0.5, color="rgba(255,255,255,0.19)")),
    text=grp_data["influence"].round(1),
    textposition="outside",
    textfont=dict(color="#e0e8ff"),
))
fig1.update_layout(
    title=dict(text="📊 Total Influence by Semantic Group (Correct-Prediction Graph)",
               font=dict(size=15, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    xaxis=dict(title="Summed Influence", color="#8899bb", gridcolor="#1a2233"),
    yaxis=dict(color="#c0ccdd", tickfont=dict(size=11)),
    height=480, margin=dict(l=180, r=80, t=60, b=50),
    font=dict(color="#c0ccdd"),
)
fig1.show()

# ════════════════════════════════════════════════════════════
# VIZ 2 — LAYER × GROUP HEATMAP
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 2: Layer × Semantic Group Heatmap...")

layer_group = df_nodes[df_nodes.layer >= 0].groupby(
    ["layer", "group"])["influence"].sum().reset_index()
pivot = layer_group.pivot(index="group", columns="layer", values="influence").fillna(0)

# Reorder groups
order = ["p1_bath","p1_bed","p1_number","p1_id",
         "p2_bath","p2_bed","p2_number","p2_id",
         "structural","embedding","logit"]
pivot = pivot.reindex([g for g in order if g in pivot.index])

fig2 = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[f"L{c}" for c in pivot.columns],
    y=[GROUP_LABELS.get(g, g) for g in pivot.index],
    colorscale="Viridis",
    hovertemplate="<b>%{y}</b><br>Layer %{x}<br>Influence: %{z:.2f}<extra></extra>",
    colorbar=dict(title="Influence", tickfont=dict(color="#c0ccdd"),
                  titlefont=dict(color="#c0ccdd")),
))
fig2.update_layout(
    title=dict(text="🔥 Influence Heatmap: Layer × Semantic Group",
               font=dict(size=15, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    height=420, margin=dict(l=200, r=50, t=60, b=60),
    xaxis=dict(color="#8899bb", tickangle=-45, tickfont=dict(size=9)),
    yaxis=dict(color="#c0ccdd"),
    font=dict(color="#c0ccdd"),
)
fig2.show()

# ════════════════════════════════════════════════════════════
# VIZ 3 — LAYER INFLUENCE PROFILE (line + bar)
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 3: Layer Influence Profile...")

layer_total = df_nodes[df_nodes.layer >= 0].groupby("layer")["influence"].agg(
    ["sum","count","mean"]).reset_index()
layer_total.columns = ["layer","total","count","mean"]

fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(go.Bar(
    x=layer_total["layer"], y=layer_total["total"],
    name="Total Influence", marker_color="#0099ff", opacity=0.7,
    hovertemplate="Layer %{x}<br>Total: %{y:.1f}<extra></extra>",
), secondary_y=False)
fig3.add_trace(go.Scatter(
    x=layer_total["layer"], y=layer_total["count"],
    mode="lines+markers", name="Node Count",
    line=dict(color="#ff6633", width=2),
    marker=dict(size=5),
    hovertemplate="Layer %{x}<br>Nodes: %{y}<extra></extra>",
), secondary_y=True)
fig3.update_layout(
    title=dict(text="📈 Per-Layer Influence Profile (Total Influence + Node Count)",
               font=dict(size=15, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    height=420, legend=dict(bgcolor="#0d1525", font=dict(color="#c0ccdd")),
    xaxis=dict(title="Layer", color="#8899bb", gridcolor="#1a2233"),
    font=dict(color="#c0ccdd"),
)
fig3.update_yaxes(title_text="Total Influence", color="#0099ff", secondary_y=False,
                  gridcolor="#1a2233")
fig3.update_yaxes(title_text="Node Count", color="#ff6633", secondary_y=True)
fig3.show()

# ════════════════════════════════════════════════════════════
# VIZ 4 — TOKEN POSITION INFLUENCE STRIP
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 4: Token Position Influence Strip...")

ctx_sum = df_nodes.groupby("ctx_idx")["influence"].sum().reset_index()
ctx_sum["token"] = ctx_sum["ctx_idx"].map(lambda i: TOKEN_MAP.get(i, ("?","?"))[0])
ctx_sum["group"] = ctx_sum["ctx_idx"].map(lambda i: TOKEN_MAP.get(i, ("?","structural"))[1])
ctx_sum["color"] = ctx_sum["group"].map(GROUP_COLORS)
ctx_sum = ctx_sum.sort_values("ctx_idx")

fig4 = go.Figure()
fig4.add_trace(go.Bar(
    x=ctx_sum["ctx_idx"],
    y=ctx_sum["influence"],
    marker_color=ctx_sum["color"],
    text=ctx_sum["token"],
    textposition="outside",
    textfont=dict(size=9, color="#c0ccdd"),
    hovertemplate="<b>%{text}</b><br>ctx %{x}<br>Influence: %{y:.1f}<extra></extra>",
))
# Annotate key tokens
for _, row in ctx_sum[ctx_sum["group"].isin(["p1_bath","p1_bed","p2_bath","p2_bed"])].iterrows():
    fig4.add_annotation(
        x=row.ctx_idx, y=row.influence + 15,
        text=f"<b>{row.token}</b>",
        font=dict(color=row.color, size=10), showarrow=False,
    )
fig4.update_layout(
    title=dict(text="🪙 Influence by Token Position (every ctx slot in the prompt)",
               font=dict(size=15, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    height=450, margin=dict(l=50, r=50, t=70, b=50),
    xaxis=dict(title="Token Position (ctx_idx)", color="#8899bb",
               tickvals=list(range(32)),
               ticktext=[TOKEN_MAP.get(i,("?","?"))[0] for i in range(32)],
               tickangle=-45, tickfont=dict(size=8), gridcolor="#1a2233"),
    yaxis=dict(title="Summed Influence", color="#8899bb", gridcolor="#1a2233"),
    font=dict(color="#c0ccdd"),
)
fig4.show()

# ════════════════════════════════════════════════════════════
# VIZ 5 — TOP 50 NODES TABLE (interactive)
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 5: Top 50 Nodes Table...")

top50 = df_nodes.nlargest(50, "influence")[
    ["node_id","layer","ctx_idx","token","group","influence","activation","clerp","feature"]
].copy()
top50["clerp"] = top50["clerp"].fillna("—").replace("", "—")
top50["influence"] = top50["influence"].round(4)
top50["activation"] = top50["activation"].round(4)

fill_colors = [[GROUP_COLORS.get(g, "#334455") + "44" for g in top50["group"]]]

fig5 = go.Figure(go.Table(
    header=dict(
        values=["<b>Node ID</b>","<b>Layer</b>","<b>ctx</b>","<b>Token</b>",
                "<b>Group</b>","<b>Influence</b>","<b>Activation</b>","<b>Clerp</b>","<b>Feature</b>"],
        fill_color="#1a2a3a", font=dict(color="#e0e8ff", size=12),
        align="center", height=30,
    ),
    cells=dict(
        values=[top50[c] for c in ["node_id","layer","ctx_idx","token","group",
                                    "influence","activation","clerp","feature"]],
        fill_color=[["#0d1525" if i%2==0 else "#0f1e2e" for i in range(50)]],
        font=dict(color="#c0ccdd", size=11),
        align=["left","center","center","center","center","right","right","left","left"],
        height=25,
    )
))
fig5.update_layout(
    title=dict(text="📋 Top 50 Most Influential Nodes (by influence score)",
               font=dict(size=15, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", height=900, margin=dict(l=10, r=10, t=60, b=10),
)
fig5.show()

# ════════════════════════════════════════════════════════════
# VIZ 6 — FULL GRAPH NETWORK (top 300 nodes by influence)
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 6: Full Attribution Network (top 300 nodes)...")

TOP_N_FULL = 300
top_ids = set(df_nodes.nlargest(TOP_N_FULL, "influence")["node_id"])
G_full = build_nx_subgraph(top_ids, max_edges=3000)

# Use layered x-pos + influence y-pos layout
pos_full = {}
for nid in G_full.nodes():
    nd = G_full.nodes[nid]
    layer = float(nd.get("layer", 0))
    ctx   = float(nd.get("ctx_idx", 16))
    # x = layer position, y = token position (spread)
    pos_full[nid] = (layer + np.random.normal(0, 0.3),
                     ctx   + np.random.normal(0, 0.2))

# Draw edges colored by weight
edge_traces = []
link_sub = df_links[
    df_links.source.isin(top_ids) & df_links.target.isin(top_ids)
].assign(abs_w=lambda x: x["weight"].abs()).nlargest(2000, "abs_w").drop(columns="abs_w")

pos_edges = []
for _, e in link_sub.iterrows():
    if e.source in pos_full and e.target in pos_full:
        x0, y0 = pos_full[e.source]; x1, y1 = pos_full[e.target]
        col = f"rgba(0,200,100,0.3)" if e.weight > 0 else f"rgba(255,50,50,0.3)"
        pos_edges.append((x0, x1, y0, y1, col, e.weight))

for x0, x1, y0, y1, col, w in pos_edges:
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode="lines", line=dict(width=max(0.3, abs(w)*2), color=col),
        hoverinfo="none", showlegend=False
    ))

nx_x = [pos_full[n][0] for n in G_full.nodes()]
nx_y = [pos_full[n][1] for n in G_full.nodes()]
nx_colors = [G_full.nodes[n].get("color", "#aaa") for n in G_full.nodes()]
nx_sizes  = [max(5, min(25, G_full.nodes[n].get("influence",0)*12)) for n in G_full.nodes()]
nx_text   = [
    f"<b>{G_full.nodes[n].get('token','?')}</b><br>"
    f"Layer {G_full.nodes[n].get('layer','?')} | ctx {G_full.nodes[n].get('ctx_idx','?')}<br>"
    f"Influence: {G_full.nodes[n].get('influence',0):.4f}<br>"
    f"Activation: {G_full.nodes[n].get('activation',0):.4f}<br>"
    f"Clerp: {G_full.nodes[n].get('clerp','') or '—'}"
    for n in G_full.nodes()
]

node_trace_full = go.Scatter(
    x=nx_x, y=nx_y, mode="markers",
    marker=dict(color=nx_colors, size=nx_sizes,
                line=dict(width=0.8, color="rgba(255,255,255,0.31)"),
                opacity=0.9),
    text=nx_text, hoverinfo="text",
)

# Add token position guidelines
shapes = []
for ctx_i, (tok, grp) in TOKEN_MAP.items():
    base_col = GROUP_COLORS.get(grp, "#333344")
    # Convert hex to rgba for plotly shape compatibility
    r, g, b = int(base_col[1:3],16), int(base_col[3:5],16), int(base_col[5:7],16)
    shapes.append(dict(type="line", x0=-1, x1=37, y0=ctx_i, y1=ctx_i,
                       line=dict(color=f"rgba({r},{g},{b},0.18)", width=0.5, dash="dot")))

# Layer gridlines
for l in range(37):
    shapes.append(dict(type="line", x0=l, x1=l, y0=-1, y1=33,
                       line=dict(color="rgba(26,34,51,0.9)", width=0.5)))

fig6 = go.Figure(edge_traces + [node_trace_full])
fig6.update_layout(
    title=dict(text=f"🕸️ Full Attribution Graph — Top {TOP_N_FULL} Nodes<br>"
               "<sup>X=Layer (0→35) | Y=Token Position | Green edges=positive weight | Red=negative</sup>",
               font=dict(size=14, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    height=850,
    xaxis=dict(title="Layer", color="#8899bb", gridcolor="#1a2233",
               tickvals=list(range(36)), ticktext=[str(i) for i in range(36)],
               range=[-1, 37]),
    yaxis=dict(title="Token Position (ctx_idx)", color="#8899bb",
               tickvals=list(TOKEN_MAP.keys()),
               ticktext=[f"{i}: {TOKEN_MAP[i][0]}" for i in TOKEN_MAP],
               range=[-1, 33], tickfont=dict(size=9)),
    shapes=shapes,
    showlegend=False,
    font=dict(color="#c0ccdd"),
    margin=dict(l=130, r=30, t=80, b=50),
)

# Legend annotation
for grp, col in GROUP_COLORS.items():
    fig6.add_annotation(x=38, y=-99, text="", showarrow=False)  # dummy
fig6.show()

# ════════════════════════════════════════════════════════════
# VIZ 7 — NARROWED GRAPH: key semantic nodes only
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 7: Narrowed Semantic Graph (bath/bed/number tokens)...")

SEMANTIC_CTXS = {6, 7, 10, 11, 18, 19, 22, 23}   # all bath+bed+number positions
TOP_PER_CTX   = 15                                  # top N nodes per ctx slot

semantic_nodes = pd.concat([
    df_nodes[df_nodes.ctx_idx == c].nlargest(TOP_PER_CTX, "influence")
    for c in SEMANTIC_CTXS
])
logit_nodes = df_nodes[df_nodes.is_logit]
sem_ids = set(semantic_nodes.node_id) | set(logit_nodes.node_id)

G_sem = build_nx_subgraph(sem_ids, max_edges=1000)

# Force layout: x=layer, y spread by token group
sem_pos = {}
group_y = {"p1_bath":26,"p1_number_bath":24,"p1_bed":20,"p1_number_bed":18,
           "p2_bath":10,"p2_number_bath":8,"p2_bed":4,"p2_number_bed":2,
           "logit":14}
ctx_y_map = {7:26, 6:24, 11:20, 10:18, 19:10, 18:8, 23:4, 22:2}

jitter_counter = collections.defaultdict(int)
for nid in G_sem.nodes():
    nd = G_sem.nodes[nid]
    lyr = float(nd.get("layer", 0))
    ctx = int(nd.get("ctx_idx", 16))
    base_y = ctx_y_map.get(ctx, 14)
    jitter = jitter_counter[(lyr, ctx)] * 0.6
    jitter_counter[(lyr, ctx)] += 1
    sem_pos[nid] = (lyr, base_y + jitter)

sem_edge_x, sem_edge_y = [], []
sem_edge_cols = []
sem_link_df = df_links[
    df_links.source.isin(sem_ids) & df_links.target.isin(sem_ids)
]
for _, e in sem_link_df.iterrows():
    if e.source in sem_pos and e.target in sem_pos:
        x0, y0 = sem_pos[e.source]; x1, y1 = sem_pos[e.target]
        sem_edge_x += [x0, x1, None]
        sem_edge_y += [y0, y1, None]

sem_edge_trace = go.Scatter(
    x=sem_edge_x, y=sem_edge_y, mode="lines",
    line=dict(width=0.8, color="#004466"),
    hoverinfo="none"
)

sx = [sem_pos[n][0] for n in G_sem.nodes()]
sy = [sem_pos[n][1] for n in G_sem.nodes()]
sc = [G_sem.nodes[n].get("color", "#aaa") for n in G_sem.nodes()]
ss = [max(8, min(35, G_sem.nodes[n].get("influence", 0) * 18)) for n in G_sem.nodes()]
st = [
    f"<b>{G_sem.nodes[n].get('token','?')}</b><br>"
    f"Layer {G_sem.nodes[n].get('layer','?')} | ctx {G_sem.nodes[n].get('ctx_idx','?')}<br>"
    f"Influence: {G_sem.nodes[n].get('influence',0):.4f}<br>"
    f"Activation: {G_sem.nodes[n].get('activation',0):.4f}<br>"
    f"Clerp: {G_sem.nodes[n].get('clerp','') or '—'}<br>"
    f"Feature ID: {G_sem.nodes[n].get('feature','?')}"
    for n in G_sem.nodes()
]

sem_node_trace = go.Scatter(
    x=sx, y=sy, mode="markers+text",
    marker=dict(color=sc, size=ss, line=dict(width=1, color="rgba(255,255,255,0.38)"), opacity=0.92),
    text=[G_sem.nodes[n].get("token","") if G_sem.nodes[n].get("influence",0) > 0.5 else "" for n in G_sem.nodes()],
    textposition="top center", textfont=dict(size=7, color="#c0ccdd"),
    hovertext=st, hoverinfo="text",
)

fig7 = go.Figure([sem_edge_trace, sem_node_trace])

# Add token lane labels
for ctx_i, y_val in ctx_y_map.items():
    tok, grp = TOKEN_MAP.get(ctx_i, ("?","structural"))
    fig7.add_annotation(
        x=-1.5, y=y_val + 1.5, text=f"<b>{tok}</b>",
        font=dict(color=GROUP_COLORS.get(grp, "#aaa"), size=11),
        showarrow=False, xanchor="right",
    )
    r2, g2, b2 = int(GROUP_COLORS.get(grp,"#111111")[1:3],16), \
                 int(GROUP_COLORS.get(grp,"#111111")[3:5],16), \
                 int(GROUP_COLORS.get(grp,"#111111")[5:7],16)
    fig7.add_shape(type="rect",
        x0=-1, x1=36, y0=y_val-0.5, y1=y_val+4,
        fillcolor=f"rgba({r2},{g2},{b2},0.06)",
        line=dict(width=0),
    )

fig7.update_layout(
    title=dict(text="🔬 Narrowed Semantic Graph — Bathroom / Bedroom / Number Tokens Only<br>"
               "<sup>X=Layer (0→35) | Y=Token Lane | Size ∝ Influence | Hover for full details</sup>",
               font=dict(size=14, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    height=820,
    xaxis=dict(title="Layer", color="#8899bb", gridcolor="#1a2233",
               tickvals=list(range(36)), ticktext=[str(i) for i in range(36)],
               range=[-3, 37]),
    yaxis=dict(showticklabels=False, color="#8899bb", range=[-2, 32]),
    showlegend=False,
    font=dict(color="#c0ccdd"),
    margin=dict(l=120, r=30, t=90, b=50),
)
fig7.show()

# ════════════════════════════════════════════════════════════
# VIZ 8 — EDGE WEIGHT DISTRIBUTION
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 8: Edge Weight Distribution...")

sample_weights = df_links["weight"].sample(min(50000, len(df_links)), random_state=42)

fig8 = go.Figure()
fig8.add_trace(go.Histogram(
    x=sample_weights[sample_weights > 0],
    name="Positive (→ correct)", nbinsx=80,
    marker_color="#00bb66", opacity=0.8,
))
fig8.add_trace(go.Histogram(
    x=sample_weights[sample_weights < 0],
    name="Negative (→ wrong)", nbinsx=80,
    marker_color="#ee3355", opacity=0.8,
))
fig8.update_layout(
    title=dict(text="⚖️ Edge Weight Distribution (sample of 50k edges)",
               font=dict(size=15, color="#e0e8ff")),
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    barmode="overlay",
    xaxis=dict(title="Edge Weight", color="#8899bb", gridcolor="#1a2233"),
    yaxis=dict(title="Count", color="#8899bb", gridcolor="#1a2233"),
    height=420, legend=dict(bgcolor="#0d1525", font=dict(color="#c0ccdd")),
    font=dict(color="#c0ccdd"),
)
fig8.show()

# ════════════════════════════════════════════════════════════
# VIZ 9 — ACTIVATION vs INFLUENCE SCATTER
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 9: Activation vs Influence Scatter...")

scatter_df = df_nodes[df_nodes.layer >= 0].copy()
scatter_df["label"] = scatter_df["group"].map(GROUP_LABELS)

fig9 = px.scatter(
    scatter_df,
    x="activation", y="influence",
    color="group", color_discrete_map=GROUP_COLORS,
    opacity=0.6, size_max=10,
    hover_data=["node_id","layer","ctx_idx","token","clerp","feature"],
    labels={"group": "Semantic Group"},
    title="🔵 Activation vs Influence — All Nodes",
    template="plotly_dark",
)
fig9.update_traces(marker=dict(size=4))
fig9.update_layout(
    paper_bgcolor="#0a0e1a", plot_bgcolor="#0a0e1a",
    height=500, font=dict(color="#c0ccdd"),
    legend=dict(bgcolor="#0d1525", font=dict(color="#c0ccdd")),
)
fig9.show()

# ════════════════════════════════════════════════════════════
# VIZ 10 — RADIAL CIRCUIT OVERVIEW  (sunburst)
# ════════════════════════════════════════════════════════════
print("🎨 Rendering Viz 10: Sunburst — Group → Layer → Feature...")

sb_df = df_nodes[df_nodes.layer >= 0].copy()
sb_df["layer_str"] = "L" + sb_df["layer"].astype(str)
# Keep only top nodes to avoid browser overload
sb_top = sb_df.nlargest(500, "influence")
sb_top["group_label"] = sb_top["group"].map(GROUP_LABELS)
sb_top["node_label"] = sb_top.apply(
    lambda r: f"{r['token']}@L{r['layer']}" , axis=1
)

fig10 = px.sunburst(
    sb_top,
    path=["group_label", "layer_str", "node_label"],
    values="influence",
    color="group",
    color_discrete_map=GROUP_COLORS,
    title="☀️ Sunburst: Group → Layer → Node (top 500 by influence)",
    branchvalues="total",
)
fig10.update_layout(
    paper_bgcolor="#0a0e1a",
    height=750, font=dict(color="#c0ccdd", size=11),
    title_font=dict(size=15, color="#e0e8ff"),
)
fig10.update_traces(
    textfont_size=10,
    insidetextorientation="radial",
)
fig10.show()

print("\n✅ All 10 visualizations rendered.")
print("━"*60)
print("SUMMARY:")
print(f"  Total nodes:       {len(df_nodes):>8,}")
print(f"  Total links:       {len(df_links):>8,}")
print(f"  Top influence:     {df_nodes.influence.max():>8.4f}  (node: {df_nodes.nlargest(1,'influence').node_id.values[0]})")
print(f"  Structural share:  {df_nodes[df_nodes.group=='structural'].influence.sum()/df_nodes.influence.sum()*100:>7.1f}%")
sem_ctxs = {7,11,19,23}
sem_inf = df_nodes[df_nodes.ctx_idx.isin(sem_ctxs)].influence.sum()
print(f"  Bath/Bed share:    {sem_inf/df_nodes.influence.sum()*100:>7.1f}%")
print(f"  Nodes with clerp:  {(df_nodes.clerp != '').sum():>8,}")

⏳ Loading JSON graph (~345MB)...
✅ Loaded: 5,271 nodes | 3,265,350 links

📊 Node type breakdown:
feature_type
cross layer transcoder      4499
embedding                     32
logit                          2
mlp reconstruction error     738

📊 Semantic group breakdown:
            count  total_influence  mean_influence
group                                             
structural   3079         2006.571           0.652
p1_bath       444          273.962           0.617
output        450          268.766           0.597
p1_bed        337          223.304           0.663
p2_id         213          138.566           0.651
p1_number     169          112.180           0.664
p2_number     156          110.213           0.706
p1_id         157          101.189           0.645
p2_bed        129           89.903           0.697
p2_bath       104           72.449           0.697
embedding      32            6.477           0.202
logit           1            0.000           0.000

🎨 Rendering Vi

🎨 Rendering Viz 2: Layer × Semantic Group Heatmap...


🎨 Rendering Viz 3: Layer Influence Profile...


🎨 Rendering Viz 4: Token Position Influence Strip...


🎨 Rendering Viz 5: Top 50 Nodes Table...


...

🎨 Rendering Viz 7: Narrowed Semantic Graph (bath/bed/number tokens)...


🎨 Rendering Viz 8: Edge Weight Distribution...


🎨 Rendering Viz 9: Activation vs Influence Scatter...


🎨 Rendering Viz 10: Sunburst — Group → Layer → Feature...



✅ All 10 visualizations rendered.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SUMMARY:
  Total nodes:          5,271
  Total links:       3,265,350
  Top influence:       0.8000  (node: 2_65146_19)
  Structural share:     59.0%
  Bath/Bed share:       19.4%
  Nodes with clerp:         2
